In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as img

import cv2
import itertools
import pathlib
import warnings
from PIL import Image
from random import randint
warnings.filterwarnings('ignore')

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef as MCC
from sklearn.metrics import balanced_accuracy_score as BAS
from sklearn.metrics import classification_report, confusion_matrix


from tensorflow import keras
from keras import layers
import tensorflow as tf
#import tensorflow_addons as tfa
from tensorflow.keras.preprocessing import image_dataset_from_directory
##from keras.utils.vis_utils import plot_model
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.layers import Conv2D, Flatten
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator as IDG
from tensorflow.keras.layers import SeparableConv2D, BatchNormalization, GlobalAveragePooling2D

from distutils.dir_util import copy_tree, remove_tree

import os
#print(os.listdir("../input/alzheimer-mri-dataset/Dataset"))

print("TensorFlow Version:", tf.__version__)

In [ ]:
import tensorflow as tf
from keras.datasets import mnist
import cv2
import os
import pathlib
from keras.layers import Conv2D, Conv2DTranspose, Dropout, Dense, Reshape, LayerNormalization, LeakyReLU
from keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score, recall_score, precision_score

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42, stratify=labels)
X_train.shape,X_test.shape, y_train.shape,y_test.shape

In [ ]:
images_train = X_train
images_test = X_test

labels_train = y_train
labels_test = y_test

images_train.shape,images_test.shape, labels_train.shape,labels_test.shape

In [ ]:
import tensorflow as tf
from keras.datasets import mnist
import cv2
import os
import pathlib
from keras.layers import Conv2D, Conv2DTranspose, Dropout, Dense, Reshape, LayerNormalization, LeakyReLU
from keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score, recall_score, precision_score

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNet, ResNet50
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Input shape
input_shape = (128, 128, 3)

# Load pre-trained ResNet50 model
mobilenet_base = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=input_shape
)

# Freeze MobileNet layers
mobilenet_base.trainable = False

# Define input
input_data = Input(shape=input_shape, name='input_data')

# Do NOT rescale here because preprocessing is already done
x = mobilenet_base(input_data, training=False)

# Classification head
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='selu')(x)

# Since y_train_one_hot is one-hot encoded, use softmax
output_layer = Dense(2, activation='softmax')(x)

# Create model
model1 = Model(inputs=input_data, outputs=output_layer)

# Compile model
model1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Checkpoint callback
checkpoint_callback = ModelCheckpoint(
    filepath='best_models1.keras',
    save_weights_only=False,
    save_freq='epoch',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# Early stopping callback
early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    verbose=1
)

callbacks = [checkpoint_callback, early_stopping_callback]

# Train model
history = model1.fit(
    images_train,
    y_train_one_hot,
    epochs=100,
    callbacks=callbacks,
    validation_split=0.2,
    shuffle=True
)

In [ ]:
# ============================================================
# TAC-CONCORD + pretrained ResNet50 for CIFAR-10
# Certified-only randomized defense across (sigma, radius)
#
# Assumes you already have:
#   X_train, y_train, X_test, y_test
#
# Uses your classifier:
#   ResNet50(weights='imagenet', include_top=False, input_shape=(96,96,3))
#   GlobalAveragePooling2D
#   Dense(256, relu)
#   Dropout(0.3)
#   Dense(10)
#
# Main outputs:
#   - trained TAC-CONCORD-ResNet50 model
#   - certified accuracy table across sigma and radius
#
# ============================================================

import os
import math
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.stats import beta as beta_dist
from scipy.stats import norm as normal_dist

from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


# ============================================================
# 0. Reproducibility and data preparation
# ============================================================

SEED = 123
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

NUM_CLASSES = 2

'''def normalize_images_if_needed(X):
    X = X.astype("float32")
    if np.max(X) > 2.0:
        X = X / 255.0
    return X

def to_onehot(y, num_classes=10):
    y = np.asarray(y)
    if y.ndim == 2:
        return y.astype("float32")
    return tf.keras.utils.to_categorical(y.reshape(-1), num_classes).astype("float32")
'''

def labels_from_y(y):
    y = np.asarray(y)
    if y.ndim == 2:
        return np.argmax(y, axis=1).astype("int64")
    return y.reshape(-1).astype("int64")

X_train = images_train
    #normalize_images_if_needed(X_train)
X_test = X_test #normalize_images_if_needed(X_test)

y_train_oh = y_train_one_hot #to_onehot(y_train, NUM_CLASSES)
y_test_oh = y_test_one_hot  ##to_onehot(y_test, NUM_CLASSES)
y_test_labels = labels_from_y(y_test_one_hot)

INPUT_SHAPE = X_train.shape[1:]
TRAIN_MEAN = np.mean(X_train, axis=0, keepdims=True).astype("float32")

#print("X_train:", X_train.shape, X_train.dtype, "range:", float(X_train.min()), float(X_train.max()))
#print("X_test :", X_test.shape, X_test.dtype, "range:", float(X_test.min()), float(X_test.max()))
#print("y_train:", y_train_oh.shape)
#print("Input shape:", INPUT_SHAPE)


# ============================================================
# 1. Exact orthogonal color-tangent layer
# ============================================================

class OrthogonalColorTangent(layers.Layer):
    """
    Learnable 3x3 orthogonal color transformation using Cayley transform.

    If A is skew-symmetric:
        O = (I + A)^(-1)(I - A)

    is orthogonal, so it preserves L2 distances exactly.
    """

    def __init__(self, channels=3, init_scale=0.0, **kwargs):
        super().__init__(**kwargs)
        self.channels = int(channels)
        self.init_scale = float(init_scale)

    def build(self, input_shape):
        self.raw_A = self.add_weight(
            name="raw_skew_color_matrix",
            shape=(self.channels, self.channels),
            initializer=tf.keras.initializers.RandomNormal(stddev=self.init_scale),
            trainable=True,
        )

    def get_orthogonal_matrix(self):
        A = self.raw_A - tf.transpose(self.raw_A)
        I = tf.eye(self.channels, dtype=A.dtype)
        O = tf.linalg.solve(I + A, I - A)
        return O

    def call(self, x):
        O = self.get_orthogonal_matrix()
        return tf.einsum("bhwc,cd->bhwd", x, O)


# ============================================================
# 2. Certified stochastic coalescent flow
# ============================================================

class TACCoalescentFlow(layers.Layer):
    """
    Certified OU-style stochastic coalescent flow.

    Per step:
        x_{k+1} = mean + a * (x_k - mean) + step_noise * eps

    with:
        a = exp(-lambda_cert * dt)

    Then an optional orthogonal color-tangent map is applied.
    This preserves L2 distances and therefore does not weaken the certificate.

    The terminal distribution is Gaussian around a contracted version of x.
    The certified radius later uses the exact coalescence/TV formula.
    """

    def __init__(
        self,
        train_mean,
        T=1.0,
        steps=8,
        lambda_cert=1.0,
        use_color_tangent=True,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.T = float(T)
        self.steps = int(steps)
        self.lambda_cert = float(lambda_cert)
        self.use_color_tangent = bool(use_color_tangent)

        self.dt = self.T / self.steps
        self.train_mean_np = train_mean.astype("float32")
        self.train_mean = tf.constant(self.train_mean_np, dtype=tf.float32)

        if self.use_color_tangent:
            self.color_tangent = OrthogonalColorTangent(
                channels=int(train_mean.shape[-1]),
                init_scale=0.0,
                name="orthogonal_color_tangent",
            )
        else:
            self.color_tangent = None

    def step_noise_std_and_a(self, sigma):
        sigma = tf.cast(sigma, tf.float32)

        if self.lambda_cert > 0:
            a = tf.exp(tf.constant(-self.lambda_cert * self.dt, dtype=tf.float32))
            var = sigma ** 2 * (1.0 - a ** 2) / (2.0 * self.lambda_cert)
            return tf.sqrt(tf.maximum(var, 1e-12)), a

        a = tf.constant(1.0, dtype=tf.float32)
        var = sigma ** 2 * self.dt
        return tf.sqrt(tf.maximum(var, 1e-12)), a

    def call(self, x, sigma, training=False):
        x = tf.cast(x, tf.float32)
        sigma = tf.cast(sigma, tf.float32)

        step_std, a = self.step_noise_std_and_a(sigma)

        z = x
        for _ in range(self.steps):
            eps = tf.random.normal(tf.shape(z), dtype=z.dtype)

            z = self.train_mean + a * (z - self.train_mean) + step_std * eps

            if self.color_tangent is not None:
                centered = z - self.train_mean
                centered = self.color_tangent(centered)
                z = self.train_mean + centered

        return z


# ============================================================
# 3. Your pretrained ResNet50 classifier
# ============================================================

class ResNet50CIFARClassifier(tf.keras.Model):
    """
    Your requested classifier:

        base_model = ResNet50(
            weights='imagenet',
            include_top=False,
            input_shape=(96,96,3)
        )

        GlobalAveragePooling2D()
        Dense(256, activation='relu')
        Dropout(0.3)
        Dense(10)

    This model returns logits. TAC-CONCORD applies softmax outside.

    If your X_train is 32x32, this internally resizes to 96x96.
    If your X_train is already 96x96, this still works.
    """

    def __init__(
        self,
        num_classes=2,
        resnet_input_shape=(128, 128, 3),
        dense_units=256,
        dropout_rate=0.3,
        train_backbone=False,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.num_classes = int(num_classes)
        self.resnet_input_shape = tuple(resnet_input_shape)
        self.dense_units = int(dense_units)
        self.dropout_rate = float(dropout_rate)
        self.train_backbone = bool(train_backbone)

        self.base_model = ResNet50(
            weights="imagenet",
            include_top=False,
            input_shape=self.resnet_input_shape,
        )

        self.base_model.trainable = self.train_backbone

        self.gap = layers.GlobalAveragePooling2D(name="global_average_pooling")
        self.fc = layers.Dense(256, activation="relu", name="dense_256")
        self.dropout = layers.Dropout(0.3, name="dropout_03")
        self.logits_layer = layers.Dense(num_classes, activation="sigmoid", name="class_logits")

    def call(self, x, training=False):
        x = tf.cast(x, tf.float32)

        # Classifier-side deterministic post-processing.
        x = tf.clip_by_value(x, 0.0, 1.0)

        # Your ResNet50 expects 96x96 input.
        x = tf.image.resize(x, (96, 96), method="bilinear")

        # ImageNet ResNet50 preprocessing.
        x = x * 255.0
        x = preprocess_input(x)

        backbone_training = bool(training and self.base_model.trainable)

        x = self.base_model(x, training=backbone_training)
        x = self.gap(x)
        x = self.fc(x)
        x = self.dropout(x, training=training)

        logits = self.logits_layer(x)
        return logits


# ============================================================
# 4. TAC-CONCORD model with pretrained ResNet50
# ============================================================

class TACConcordResNet50Model(tf.keras.Model):
    """
    Full model:

        image x
          -> TAC stochastic coalescent flow
          -> pretrained ResNet50 classifier
          -> class probabilities

    This model uses custom train_step/test_step to train with:
        - CE loss
        - consistency loss across two stochastic trajectories
        - probability-gap loss for certification
    """

    def __init__(
        self,
        input_shape,
        train_mean,
        num_classes=10,
        T=1.0,
        steps=8,
        lambda_cert=1.0,
        sigma_train_values=(0.12, 0.25, 0.50),
        eval_sigma=0.25,
        consistency_weight=0.10,
        gap_weight=0.25,
        target_gap=0.50,
        train_backbone=False,
        use_color_tangent=True,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.num_classes = int(num_classes)
        self.T = float(T)
        self.steps = int(steps)
        self.lambda_cert = float(lambda_cert)

        self.consistency_weight = float(consistency_weight)
        self.gap_weight = float(gap_weight)
        self.target_gap = float(target_gap)

        self.sigma_train_values = tf.constant(sigma_train_values, dtype=tf.float32)
        self.eval_sigma = tf.Variable(float(eval_sigma), trainable=False, dtype=tf.float32)

        self.flow = TACCoalescentFlow(
            train_mean=train_mean,
            T=T,
            steps=steps,
            lambda_cert=lambda_cert,
            use_color_tangent=use_color_tangent,
            name="tac_coalescent_flow",
        )

        self.classifier = ResNet50CIFARClassifier(
            num_classes=num_classes,
            resnet_input_shape=(96, 96, 3),
            dense_units=256,
            dropout_rate=0.3,
            train_backbone=train_backbone,
            name="pretrained_resnet50_classifier",
        )

        # Custom metrics
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.ce_tracker = tf.keras.metrics.Mean(name="ce")
        self.consistency_tracker = tf.keras.metrics.Mean(name="consistency")
        self.margin_tracker = tf.keras.metrics.Mean(name="margin_loss")
        self.acc_metric = tf.keras.metrics.CategoricalAccuracy(name="accuracy")

    @property
    def metrics(self):
        return [
            self.loss_tracker,
            self.ce_tracker,
            self.consistency_tracker,
            self.margin_tracker,
            self.acc_metric,
        ]

    def set_eval_sigma(self, sigma):
        self.eval_sigma.assign(float(sigma))

    def sample_training_sigma(self):
        idx = tf.random.uniform(
            shape=[],
            minval=0,
            maxval=tf.shape(self.sigma_train_values)[0],
            dtype=tf.int32,
        )
        return tf.gather(self.sigma_train_values, idx)

    def call(self, inputs, training=False, sigma=None):
        if sigma is None:
            if training:
                sigma = self.sample_training_sigma()
            else:
                sigma = self.eval_sigma

        x_t = self.flow(inputs, sigma=sigma, training=training)
        logits = self.classifier(x_t, training=training)
        probs = tf.nn.softmax(logits, axis=-1)
        return probs

    def _margin_loss(self, y_true, probs):
        true_prob = tf.reduce_sum(y_true * probs, axis=-1)

        other_probs = probs - 1e6 * y_true
        max_other = tf.reduce_max(other_probs, axis=-1)

        gap = true_prob - max_other
        return tf.reduce_mean(tf.nn.relu(self.target_gap - gap))

    def train_step(self, data):
        x, y, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)
        y = tf.cast(y, tf.float32)

        # Use same sigma for both trajectories in this batch.
        sigma = self.sample_training_sigma()

        with tf.GradientTape() as tape:
            p1 = self(x, training=True, sigma=sigma)
            p2 = self(x, training=True, sigma=sigma)

            ce1 = tf.keras.losses.categorical_crossentropy(y, p1)
            ce2 = tf.keras.losses.categorical_crossentropy(y, p2)
            ce = tf.reduce_mean(0.5 * (ce1 + ce2))

            consistency = tf.reduce_mean(tf.square(p1 - p2))

            margin_loss = 0.5 * (
                self._margin_loss(y, p1) +
                self._margin_loss(y, p2)
            )

            loss = (
                ce
                + self.consistency_weight * consistency
                + self.gap_weight * margin_loss
            )

            if self.losses:
                loss += tf.add_n(self.losses)

        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        self.loss_tracker.update_state(loss)
        self.ce_tracker.update_state(ce)
        self.consistency_tracker.update_state(consistency)
        self.margin_tracker.update_state(margin_loss)
        self.acc_metric.update_state(y, p1, sample_weight=sample_weight)

        return {
            "loss": self.loss_tracker.result(),
            "ce": self.ce_tracker.result(),
            "consistency": self.consistency_tracker.result(),
            "margin_loss": self.margin_tracker.result(),
            "accuracy": self.acc_metric.result(),
            "sigma": sigma,
        }

    def test_step(self, data):
        x, y, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)
        y = tf.cast(y, tf.float32)

        p = self(x, training=False, sigma=self.eval_sigma)

        ce = tf.keras.losses.categorical_crossentropy(y, p)
        ce = tf.reduce_mean(ce)

        margin_loss = self._margin_loss(y, p)

        loss = ce + self.gap_weight * margin_loss

        self.loss_tracker.update_state(loss)
        self.ce_tracker.update_state(ce)
        self.consistency_tracker.update_state(0.0)
        self.margin_tracker.update_state(margin_loss)
        self.acc_metric.update_state(y, p, sample_weight=sample_weight)

        return {
            "loss": self.loss_tracker.result(),
            "ce": self.ce_tracker.result(),
            "consistency": self.consistency_tracker.result(),
            "margin_loss": self.margin_tracker.result(),
            "accuracy": self.acc_metric.result(),
        }


# ============================================================
# 5. Build and train TAC-CONCORD-ResNet50
# ============================================================

tf.keras.backend.clear_session()

# Certified-flow hyperparameters
T_FLOW = 1.0
FLOW_STEPS = 8
LAMBDA_CERT = 1.0

# Noise levels used during training.
SIGMA_TRAIN_VALUES = (0.12, 0.25, 0.50)

# ResNet50 is heavy. Use batch size 8 like your original code.
BATCH_SIZE = 8

model = TACConcordResNet50Model(
    input_shape=INPUT_SHAPE,
    train_mean=TRAIN_MEAN,
    num_classes=NUM_CLASSES,
    T=T_FLOW,
    steps=FLOW_STEPS,
    lambda_cert=LAMBDA_CERT,
    sigma_train_values=SIGMA_TRAIN_VALUES,
    eval_sigma=0.25,
    consistency_weight=0.10,
    gap_weight=0.25,
    target_gap=0.50,
    train_backbone=False,
    use_color_tangent=True,
    name="TAC_CONCORD_RESNET50",
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5)
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    mode="min",
    verbose=1,
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=15,
    mode="min",
    min_lr=1e-6,
    verbose=1,
)

checkpoint_path = "/kaggle/working/best_tac_concord_resnet50.weights.h5"

checkpoint = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    mode="min",
    verbose=1,
)

history = model.fit(
    X_train,
    y_train_oh,
    epochs=200,
    validation_split=0.2,
    batch_size=BATCH_SIZE,
    callbacks=[checkpoint, reduce_lr, early_stop],
    verbose=1,
)

if os.path.exists(checkpoint_path):
    model.load_weights(checkpoint_path)
    print("Loaded best weights:", checkpoint_path)


# ============================================================
# 6. Optional fine-tuning of ResNet50 backbone
# ============================================================

RUN_FINE_TUNING = False

if RUN_FINE_TUNING:
    model.classifier.base_model.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6)
    )

    fine_tune_checkpoint_path = "/kaggle/working/best_tac_concord_resnet50_finetuned.weights.h5"

    fine_tune_checkpoint = ModelCheckpoint(
        filepath=fine_tune_checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        mode="min",
        verbose=1,
    )

    history_ft = model.fit(
        X_train,
        y_train_oh,
        epochs=50,
        validation_split=0.2,
        batch_size=BATCH_SIZE,
        callbacks=[fine_tune_checkpoint, reduce_lr, early_stop],
        verbose=1,
    )

    if os.path.exists(fine_tune_checkpoint_path):
        model.load_weights(fine_tune_checkpoint_path)
        print("Loaded best fine-tuned weights:", fine_tune_checkpoint_path)


# ============================================================
# 7. Certified coalescence radius utilities
# ============================================================

def S_T_lambda(T, lambda_cert):
    T = float(T)
    lambda_cert = float(lambda_cert)

    if lambda_cert > 0:
        return (math.exp(2.0 * lambda_cert * T) - 1.0) / (2.0 * lambda_cert)

    return T

def q_tac_noncoalescence(r, sigma, T, lambda_cert):
    """
    q(r) = upper bound on probability that two coupled trajectories
    may fail to coalesce by time T.

    q(r) = 2 Phi( r / [2 sigma sqrt(S_T)] ) - 1
    """
    S = S_T_lambda(T, lambda_cert)
    denom = 2.0 * float(sigma) * math.sqrt(S)
    val = float(r) / max(denom, 1e-12)
    return 2.0 * normal_dist.cdf(val) - 1.0

def tac_certified_radius_from_gap(gap, sigma, T, lambda_cert):
    """
    Certified radius from probability gap:

        gap = pA_lower - pB_upper

    Certificate condition:

        gap > 2 q(r)

    Closed form:

        R = 2 sigma sqrt(S_T) Phi^{-1}(0.5 + gap/4)
    """
    gap = float(gap)

    if gap <= 0:
        return 0.0

    gap = min(max(gap, 0.0), 0.999999)
    arg = 0.5 + gap / 4.0

    S = S_T_lambda(T, lambda_cert)
    R = 2.0 * float(sigma) * math.sqrt(S) * normal_dist.ppf(arg)

    if not np.isfinite(R):
        return 0.0

    return max(0.0, float(R))

def binom_lower_confidence(k, n, alpha):
    """
    One-sided Clopper-Pearson lower bound.
    """
    k = int(k)
    n = int(n)

    if k <= 0:
        return 0.0

    return float(beta_dist.ppf(alpha, k, n - k + 1))

def binom_upper_confidence(k, n, alpha):
    """
    One-sided Clopper-Pearson upper bound.
    """
    k = int(k)
    n = int(n)

    if k >= n:
        return 1.0

    return float(beta_dist.ppf(1.0 - alpha, k + 1, n - k))


# ============================================================
# 8. Monte Carlo prediction counts
# ============================================================

def collect_mc_counts(
    model,
    X,
    sigma,
    n_samples,
    batch_size=16,
    num_classes=10,
    verbose=True,
):
    """
    Collect class counts under TAC-CONCORD stochastic trajectories.

    For each MC sample:
        X -> stochastic flow at sigma -> ResNet50 classifier -> predicted class

    Returns:
        counts: int array [N, num_classes]
    """
    X = X.astype("float32")
    n = len(X)

    counts = np.zeros((n, num_classes), dtype=np.int32)

    model.set_eval_sigma(sigma)

    for s in range(n_samples):
        if verbose and ((s + 1) % max(1, n_samples // 10) == 0 or s == 0):
            print(f"  MC sample {s + 1}/{n_samples} for sigma={sigma}")

        all_preds = []

        for start in range(0, n, batch_size):
            xb = X[start:start + batch_size]

            probs = model(
                tf.convert_to_tensor(xb, dtype=tf.float32),
                training=False,
                sigma=tf.constant(float(sigma), dtype=tf.float32),
            ).numpy()

            preds = np.argmax(probs, axis=1)
            all_preds.append(preds)

        all_preds = np.concatenate(all_preds, axis=0)

        rows = np.arange(n)
        counts[rows, all_preds] += 1

    return counts


# ============================================================
# 9. Certified accuracy evaluation
# ============================================================

def certify_dataset_tac_concord(
    model,
    X,
    y,
    sigma,
    radii_grid=(0.25, 0.50, 0.75, 1.00, 1.25, 1.50),
    N0=100,
    N=1000,
    alpha=0.001,
    batch_size=16,
    num_classes=10,
    T=1.0,
    lambda_cert=1.0,
    verbose=True,
):
    """
    Certified-only evaluation.

    1. N0 samples select top class A.
    2. N fresh samples estimate pA and pB.
    3. Clopper-Pearson confidence bounds.
    4. Convert probability gap into certified radius.
    5. Report certified robust accuracy at each radius.

    alpha is per-example failure probability.
    """

    X = X.astype("float32")
    labels = labels_from_y(y)
    n = len(X)

    print("\n================================================")
    print(f"Certifying sigma={sigma}, N0={N0}, N={N}, alpha={alpha}")
    print("================================================")

    print("Selection sampling...")
    counts_select = collect_mc_counts(
        model=model,
        X=X,
        sigma=sigma,
        n_samples=N0,
        batch_size=batch_size,
        num_classes=num_classes,
        verbose=verbose,
    )

    selected_class = np.argmax(counts_select, axis=1)

    print("Certification sampling...")
    counts_cert = collect_mc_counts(
        model=model,
        X=X,
        sigma=sigma,
        n_samples=N,
        batch_size=batch_size,
        num_classes=num_classes,
        verbose=verbose,
    )

    preds = selected_class.copy()
    radii = np.zeros(n, dtype=np.float64)
    pA_lowers = np.zeros(n, dtype=np.float64)
    pB_uppers = np.zeros(n, dtype=np.float64)
    gaps = np.zeros(n, dtype=np.float64)

    # Bonferroni split:
    # alpha/2 for pA lower,
    # alpha/(2*(K-1)) for each competitor upper.
    alpha_A = alpha / 2.0
    alpha_B_each = alpha / (2.0 * max(1, num_classes - 1))

    for i in range(n):
        A = int(selected_class[i])
        kA = int(counts_cert[i, A])

        pA_lower = binom_lower_confidence(kA, N, alpha_A)

        pB_upper = 0.0

        for c in range(num_classes):
            if c == A:
                continue

            kc = int(counts_cert[i, c])
            pB_upper_c = binom_upper_confidence(kc, N, alpha_B_each)
            pB_upper = max(pB_upper, pB_upper_c)

        gap = pA_lower - pB_upper

        R = tac_certified_radius_from_gap(
            gap=gap,
            sigma=sigma,
            T=T,
            lambda_cert=lambda_cert,
        )

        pA_lowers[i] = pA_lower
        pB_uppers[i] = pB_upper
        gaps[i] = gap
        radii[i] = R

    correct = preds == labels
    mc_clean_acc = float(np.mean(correct))

    cert_acc = {}

    for r in radii_grid:
        cert_acc[float(r)] = float(np.mean(correct & (radii >= float(r))))

    result = {
        "sigma": float(sigma),
        "T": float(T),
        "lambda_cert": float(lambda_cert),
        "N0": int(N0),
        "N": int(N),
        "alpha": float(alpha),
        "preds": preds,
        "radii": radii,
        "pA_lowers": pA_lowers,
        "pB_uppers": pB_uppers,
        "gaps": gaps,
        "mc_clean_accuracy": mc_clean_acc,
        "certified_accuracy": cert_acc,
    }

    print("\nResult for sigma =", sigma)
    print("MC clean accuracy:", mc_clean_acc)

    for r, acc in cert_acc.items():
        print(f"  CertAcc radius {r:.2f}: {acc:.4f}")

    return result

def make_cert_table(results, radii_grid):
    rows = []

    for res in results:
        row = {
            "sigma": res["sigma"],
            "T": res["T"],
            "lambda": res["lambda_cert"],
            "MC_clean_acc": res["mc_clean_accuracy"],
        }

        for r in radii_grid:
            row[f"CertAcc@{r}"] = res["certified_accuracy"][float(r)]

        rows.append(row)

    return pd.DataFrame(rows)

def best_over_sigma_table(results, radii_grid):
    row = {}

    for r in radii_grid:
        row[f"BestCertAcc@{r}"] = max(
            res["certified_accuracy"][float(r)] for res in results
        )

    return pd.DataFrame([row])


# ============================================================
# 10. Run certified evaluation
# ============================================================

RADIUS_GRID = (0.25, 0.50, 0.75, 1.00, 1.25, 1.50)
SIGMAS_TO_EVAL = (0.12, 0.25, 0.50, 1.00)

# ------------------------------------------------------------
# Pilot setting for Kaggle speed.
# For final paper-style results, use full X_test and larger N.
# ------------------------------------------------------------

CERT_SUBSET = min(1000, len(X_test))
CERT_X = X_test[:CERT_SUBSET]
CERT_Y = y_test_labels[:CERT_SUBSET]

N0_CERT = 100
N_CERT = 1000
ALPHA_CERT = 0.001

# For final stronger numbers, use:
#
# CERT_X = X_test
# CERT_Y = y_test_labels
# N0_CERT = 100
# N_CERT = 10000
# ALPHA_CERT = 0.001

all_results = []

for sig in SIGMAS_TO_EVAL:
    res = certify_dataset_tac_concord(
        model=model,
        X=CERT_X,
        y=CERT_Y,
        sigma=sig,
        radii_grid=RADIUS_GRID,
        N0=N0_CERT,
        N=N_CERT,
        alpha=ALPHA_CERT,
        batch_size=16,
        num_classes=NUM_CLASSES,
        T=T_FLOW,
        lambda_cert=LAMBDA_CERT,
        verbose=True,
    )

    all_results.append(res)

cert_table = make_cert_table(all_results, RADIUS_GRID)
best_table = best_over_sigma_table(all_results, RADIUS_GRID)

print("\n==============================")
print("TAC-CONCORD-ResNet50 certified table")
print("==============================")
display(cert_table)

print("\n==============================")
print("Best over sigma")
print("==============================")
display(best_table)

cert_table_path = "/kaggle/working/tac_concord_resnet50_cert_table.csv"
best_table_path = "/kaggle/working/tac_concord_resnet50_best_over_sigma.csv"

cert_table.to_csv(cert_table_path, index=False)
best_table.to_csv(best_table_path, index=False)

print("Saved:")
print(cert_table_path)
print(best_table_path)


# ============================================================
# 11. Optional radius distribution inspection
# ============================================================

for res in all_results:
    sig = res["sigma"]
    radii = res["radii"]

    print(f"\nSigma={sig}")
    print("Mean radius:", float(np.mean(radii)))
    print("Median radius:", float(np.median(radii)))
    print("90th percentile radius:", float(np.percentile(radii, 90)))
    print("Max radius:", float(np.max(radii)))

In [ ]:
import math
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import tensorflow as tf

TensorLike = Union[tf.Tensor, np.ndarray]


# ============================================================
# CA-NCHA helpers
# ============================================================

def labels_to_int(labels: TensorLike) -> tf.Tensor:
    """
    Accept sparse labels [B] or one-hot labels [B,C].
    Return integer labels [B].
    """
    y = tf.convert_to_tensor(labels)

    if y.shape.rank is not None and y.shape.rank >= 2:
        return tf.cast(tf.argmax(y, axis=-1), tf.int32)

    return tf.cast(tf.reshape(y, [-1]), tf.int32)


def _flat(x: tf.Tensor) -> tf.Tensor:
    return tf.reshape(x, [tf.shape(x)[0], -1])


def _flat_dot(a: tf.Tensor, b: tf.Tensor) -> tf.Tensor:
    return tf.reduce_sum(_flat(a * b), axis=1)


def _flat_l2_sq(x: tf.Tensor) -> tf.Tensor:
    return tf.reduce_sum(tf.square(_flat(x)), axis=1)


def _flat_l2_norm(x: tf.Tensor, eps: float = 1e-12) -> tf.Tensor:
    return tf.sqrt(tf.maximum(_flat_l2_sq(x), tf.cast(eps, x.dtype)))


def _normalize_l2(x: tf.Tensor, eps: float = 1e-12) -> tf.Tensor:
    n = _flat_l2_norm(x, eps)
    return x / tf.reshape(n, [-1, 1, 1, 1])


def _rms_scale(x: tf.Tensor, eps: float = 1e-12) -> tf.Tensor:
    """
    Per-sample RMS scale for image-like tensors [B,H,W,C].
    """
    return tf.sqrt(
        tf.maximum(
            tf.reduce_mean(tf.square(x), axis=[1, 2, 3], keepdims=True),
            tf.cast(eps, x.dtype)
        )
    )


def _normalize_rms(x: tf.Tensor, eps: float = 1e-12) -> tf.Tensor:
    return x / _rms_scale(x, eps)


def _smooth_box_direction(x: tf.Tensor) -> tf.Tensor:
    """
    Smooth bounded direction in [-1,1].
    This avoids directly using sign(direction).
    """
    return tf.tanh(_normalize_rms(x))


def _safe_float(x: Any, default: float = float("nan")) -> float:
    try:
        if isinstance(x, tf.Tensor):
            return float(x.numpy())
        return float(x)
    except Exception:
        return default


def project_perturbation(
    x0: tf.Tensor,
    x: tf.Tensor,
    eps: float,
    norm: str = "linf",
    input_min: float = -1.0,
    input_max: float = 1.0,
) -> tf.Tensor:
    """
    Project x into an eps-ball around x0, then clip to the valid input domain.
    """
    x0 = tf.cast(x0, tf.float32)
    x = tf.cast(x, tf.float32)

    eps_t = tf.cast(eps, x.dtype)
    delta = x - x0
    n = norm.lower()

    if n in ("linf", "l_inf", "inf"):
        delta = tf.clip_by_value(delta, -eps_t, eps_t)

    elif n in ("l2", "2"):
        dnorm = _flat_l2_norm(delta)
        factor = tf.minimum(
            tf.ones_like(dnorm),
            eps_t / tf.maximum(dnorm, tf.cast(1e-12, x.dtype))
        )
        delta = delta * tf.reshape(factor, [-1, 1, 1, 1])

    else:
        raise ValueError(f"Unsupported norm: {norm}. Use 'linf' or 'l2'.")

    return tf.clip_by_value(
        x0 + delta,
        tf.cast(input_min, x.dtype),
        tf.cast(input_max, x.dtype)
    )


def model_scores_and_probs(
    model_outputs: tf.Tensor,
    outputs_are_probabilities: bool = True,
    eps: float = 1e-8,
) -> Tuple[tf.Tensor, tf.Tensor]:
    """
    Return differentiable scores and probabilities.

    If model output is softmax probabilities:
        scores = log(probabilities)

    If model output is logits:
        scores = logits
    """
    out = tf.cast(model_outputs, tf.float32)

    if outputs_are_probabilities:
        probs = tf.clip_by_value(out, eps, 1.0)
        probs = probs / tf.reduce_sum(probs, axis=1, keepdims=True)
        scores = tf.math.log(probs)
        return scores, probs

    scores = out
    probs = tf.nn.softmax(scores, axis=1)
    return scores, probs


def _masked_scores_except(
    scores: tf.Tensor,
    labels: tf.Tensor,
    fill: float = -1e9
) -> tf.Tensor:
    labels = tf.cast(tf.reshape(labels, [-1]), tf.int32)
    c = tf.shape(scores)[-1]

    mask = tf.one_hot(
        labels,
        c,
        dtype=tf.bool,
        on_value=True,
        off_value=False
    )

    return tf.where(
        mask,
        tf.cast(fill, scores.dtype) * tf.ones_like(scores),
        scores
    )


def _topk_excluding_adaptive(
    scores: tf.Tensor,
    excluded: tf.Tensor,
    k: int = 5,
    num_classes: Optional[int] = None
) -> tf.Tensor:
    """
    Return top-k class indices excluding one class per sample.

    Untargeted:
        excluded = true label

    Targeted:
        excluded = target label

    Output:
        [B, K_eff]

    Binary case:
        returns [rival, rival], because only one valid rival exists.
    """
    if num_classes is None:
        c_static = scores.shape[-1]
        if c_static is None:
            raise ValueError(
                "Number of classes is unknown. Make sure model.output_shape[-1] is fixed."
            )
        num_classes = int(c_static)

    masked = _masked_scores_except(scores, excluded, fill=-1e9)

    valid_count = max(int(num_classes) - 1, 1)

    if valid_count <= 1:
        idx = tf.argmax(masked, axis=1, output_type=tf.int32)
        return tf.stack([idx, idx], axis=1)

    k_eff = max(2, min(int(k), valid_count))

    _, inds = tf.math.top_k(masked, k=k_eff, sorted=True)
    return tf.cast(inds, tf.int32)


def _candidate_pair_positions(k_eff: int, max_pairs: int = 6) -> List[Tuple[int, int]]:
    """
    Candidate pair positions among top-k classes.

    Example k_eff=4:
        (0,1), (0,2), (0,3), (1,2), (1,3), (2,3)
    """
    pairs: List[Tuple[int, int]] = []

    for i in range(k_eff):
        for j in range(i + 1, k_eff):
            pairs.append((i, j))

    if len(pairs) == 0:
        pairs = [(0, 1)]

    if max_pairs is not None and int(max_pairs) > 0:
        pairs = pairs[:int(max_pairs)]

    return pairs


def attack_margin_from_scores(
    scores: tf.Tensor,
    labels: tf.Tensor,
    targeted: bool = False,
    targets: Optional[tf.Tensor] = None,
) -> tf.Tensor:
    """
    Positive margin means the attack condition is satisfied.

    Untargeted:
        max_other_score - true_score

    Targeted:
        target_score - max_non_target_score
    """
    labels = tf.cast(labels, tf.int32)

    if targeted:
        if targets is None:
            raise ValueError("targets must be provided when targeted=True")

        targets = tf.cast(targets, tf.int32)

        target_score = tf.gather(
            scores,
            targets,
            axis=1,
            batch_dims=1
        )

        other = tf.reduce_max(
            _masked_scores_except(scores, targets),
            axis=1
        )

        return target_score - other

    true_score = tf.gather(
        scores,
        labels,
        axis=1,
        batch_dims=1
    )

    other = tf.reduce_max(
        _masked_scores_except(scores, labels),
        axis=1
    )

    return other - true_score


def _remove_components(
    direction: tf.Tensor,
    u: tf.Tensor,
    v: tf.Tensor
) -> tf.Tensor:
    """
    Remove L2 components along u and v.
    This keeps the final direction closer to bracket-only drift.
    """
    u_hat = _normalize_l2(u)

    v_tmp = v - tf.reshape(
        _flat_dot(v, u_hat),
        [-1, 1, 1, 1]
    ) * u_hat

    v_hat = _normalize_l2(v_tmp)

    d = direction

    d = d - tf.reshape(
        _flat_dot(d, u_hat),
        [-1, 1, 1, 1]
    ) * u_hat

    d = d - tf.reshape(
        _flat_dot(d, v_hat),
        [-1, 1, 1, 1]
    ) * v_hat

    return d

In [ ]:
# ============================================================
# CA-NCHA configuration
# ============================================================

@dataclass
class CANCHAConfig:
    # Threat model
    eps: float = 16.0 / 255.0
    norm: str = "linf"
    steps: int = 60
    alpha: float = 0.65
    restarts: int = 2
    random_start: bool = True
    targeted: bool = False

    # Input/model output
    input_min: float = -1.0
    input_max: float = 1.0
    model_outputs_are_probabilities: bool = True

    # Non-commutative primitive
    bracket_probe_scale: float = 0.35
    use_commutator_loop: bool = False
    loop_probe_scale: float = 0.25
    orthogonalize_bracket_against_first_order_fields: bool = True

    # Pure mode
    pure_bracket_mode: bool = True
    first_order_leak_start: float = 0.0
    first_order_leak_end: float = 0.0

    # Search/update stability
    try_both_bracket_signs: bool = True
    line_search_factors: Tuple[float, ...] = (1.0, 0.5, 0.25)
    fallback_to_active_field_if_bracket_degenerate: bool = True
    degenerate_bracket_rms: float = 1e-8

    # Class-adaptive pair selection
    active_pair_strategy: str = "adaptive_topk"
    active_top_k: int = 5

    # active_max_pairs:
    #   0  = use all top-k pairs
    #   6  = faster default
    #   10 = all pairs for top_k=5
    active_max_pairs: int = 6

    # active_pair_score options:
    #   "bracket_margin"   -> balanced default
    #   "lookahead_margin" -> strongest but slower
    #   "bracket_rms"      -> choose largest bracket RMS
    #   "margin"           -> choose largest active margin pressure
    active_pair_score: str = "bracket_margin"

    # Schedules/diagnostics
    step_schedule: str = "cosine"
    keep_history: bool = True


# ============================================================
# CA-NCHA attack class
# ============================================================

class ClassAdaptiveNCHA:
    """
    CA-NCHA: Class-Adaptive Non-Commutative Holonomy Attack.

    For each image and each step:
        1. Build a top-k active class set.
        2. Form candidate rival pairs from that set.
        3. Compute the finite-difference Lie bracket / commutator direction.
        4. Select the best pair adaptively.
        5. Use line search over +/- bracket direction.
    """

    def __init__(
        self,
        model: tf.keras.Model,
        config: Optional[CANCHAConfig] = None,
        **kwargs: Any
    ) -> None:
        self.model = model

        if config is None:
            config = CANCHAConfig(**kwargs)

        elif kwargs:
            data = dict(config.__dict__)
            data.update(kwargs)
            config = CANCHAConfig(**data)

        self.cfg = config

    def _num_classes(self) -> int:
        c = self.model.output_shape[-1]

        if c is None:
            raise ValueError(
                "model.output_shape[-1] is None. The number of classes must be fixed."
            )

        return int(c)

    def _scores_probs(self, x: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
        out = self.model(x, training=False)

        return model_scores_and_probs(
            out,
            self.cfg.model_outputs_are_probabilities
        )

    def _success(
        self,
        scores: tf.Tensor,
        labels: tf.Tensor,
        targets: Optional[tf.Tensor]
    ) -> tf.Tensor:
        pred = tf.argmax(scores, axis=1, output_type=tf.int32)

        if self.cfg.targeted:
            if targets is None:
                raise ValueError("targets must be provided for targeted attack")

            return tf.equal(pred, tf.cast(targets, tf.int32))

        return tf.not_equal(pred, tf.cast(labels, tf.int32))

    def _active_topk_classes(
        self,
        x: tf.Tensor,
        labels: tf.Tensor,
        targets: Optional[tf.Tensor]
    ) -> tf.Tensor:
        scores, _ = self._scores_probs(x)

        if self.cfg.targeted:
            if targets is None:
                raise ValueError("targets must be provided when targeted=True")

            excluded = tf.cast(targets, tf.int32)

        else:
            excluded = tf.cast(labels, tf.int32)

        return _topk_excluding_adaptive(
            scores=scores,
            excluded=excluded,
            k=self.cfg.active_top_k,
            num_classes=self._num_classes()
        )

    def _margin_value_for_rival(
        self,
        scores: tf.Tensor,
        labels: tf.Tensor,
        rival: tf.Tensor,
        targets: Optional[tf.Tensor],
    ) -> tf.Tensor:
        """
        Untargeted margin:
            score[rival] - score[true]

        Targeted margin:
            score[target] - score[rival]
        """
        if self.cfg.targeted:
            if targets is None:
                raise ValueError("targets must be provided when targeted=True")

            target_score = tf.gather(
                scores,
                tf.cast(targets, tf.int32),
                axis=1,
                batch_dims=1
            )

            rival_score = tf.gather(
                scores,
                tf.cast(rival, tf.int32),
                axis=1,
                batch_dims=1
            )

            return target_score - rival_score

        true_score = tf.gather(
            scores,
            tf.cast(labels, tf.int32),
            axis=1,
            batch_dims=1
        )

        rival_score = tf.gather(
            scores,
            tf.cast(rival, tf.int32),
            axis=1,
            batch_dims=1
        )

        return rival_score - true_score

    def _raw_margin_gradient(
        self,
        x: tf.Tensor,
        labels: tf.Tensor,
        rival: tf.Tensor,
        targets: Optional[tf.Tensor],
    ) -> Tuple[tf.Tensor, tf.Tensor]:
        """
        Gradient of one active class-boundary margin wrt input x.
        """
        x = tf.cast(x, tf.float32)

        with tf.GradientTape() as tape:
            tape.watch(x)

            scores, _ = self._scores_probs(x)

            margin = self._margin_value_for_rival(
                scores=scores,
                labels=labels,
                rival=rival,
                targets=targets
            )

            scalar = tf.reduce_sum(margin)

        g = tape.gradient(scalar, x)

        if g is None:
            g = tf.zeros_like(x)

        return tf.stop_gradient(g), tf.stop_gradient(margin)

    def _unit_field(
        self,
        x: tf.Tensor,
        labels: tf.Tensor,
        rival: tf.Tensor,
        targets: Optional[tf.Tensor],
    ) -> Tuple[tf.Tensor, tf.Tensor]:
        g, margin = self._raw_margin_gradient(
            x=x,
            labels=labels,
            rival=rival,
            targets=targets
        )

        return _normalize_rms(g), margin

    def _finite_difference_lie_bracket(
        self,
        x0: tf.Tensor,
        x: tf.Tensor,
        labels: tf.Tensor,
        rival_a: tf.Tensor,
        rival_b: tf.Tensor,
        targets: Optional[tf.Tensor],
    ) -> Tuple[tf.Tensor, Dict[str, tf.Tensor]]:
        """
        Approximate:

            [U_a, U_b](x) = D U_b(x) U_a(x) - D U_a(x) U_b(x)

        using finite differences of white-box class-margin gradients.
        """
        cfg = self.cfg

        h = max(float(cfg.bracket_probe_scale) * float(cfg.eps), 1e-7)
        h_t = tf.cast(h, x.dtype)

        u0, margin_a = self._unit_field(
            x=x,
            labels=labels,
            rival=rival_a,
            targets=targets
        )

        v0, margin_b = self._unit_field(
            x=x,
            labels=labels,
            rival=rival_b,
            targets=targets
        )

        xu = project_perturbation(
            x0=x0,
            x=x + h_t * _smooth_box_direction(u0),
            eps=cfg.eps,
            norm=cfg.norm,
            input_min=cfg.input_min,
            input_max=cfg.input_max
        )

        xv = project_perturbation(
            x0=x0,
            x=x + h_t * _smooth_box_direction(v0),
            eps=cfg.eps,
            norm=cfg.norm,
            input_min=cfg.input_min,
            input_max=cfg.input_max
        )

        v_at_xu, _ = self._unit_field(
            x=xu,
            labels=labels,
            rival=rival_b,
            targets=targets
        )

        u_at_xv, _ = self._unit_field(
            x=xv,
            labels=labels,
            rival=rival_a,
            targets=targets
        )

        bracket = (v_at_xu - v0) / h_t - (u_at_xv - u0) / h_t

        if cfg.orthogonalize_bracket_against_first_order_fields:
            bracket = _remove_components(bracket, u0, v0)

        info = {
            "u0": u0,
            "v0": v0,
            "margin_a": margin_a,
            "margin_b": margin_b,
            "bracket_rms": tf.reshape(_rms_scale(bracket), [-1]),
            "rival_a": rival_a,
            "rival_b": rival_b,
        }

        return tf.stop_gradient(bracket), info

    def _commutator_loop_displacement(
        self,
        x0: tf.Tensor,
        x: tf.Tensor,
        labels: tf.Tensor,
        rival_a: tf.Tensor,
        rival_b: tf.Tensor,
        targets: Optional[tf.Tensor],
    ) -> Tuple[tf.Tensor, Dict[str, tf.Tensor]]:
        """
        Four-step commutator loop:

            U_a, U_b, -U_a, -U_b

        Net displacement approximates commutator drift.
        """
        cfg = self.cfg

        h = max(float(cfg.loop_probe_scale) * float(cfg.eps), 1e-7)
        h_t = tf.cast(h, x.dtype)

        u0, margin_a = self._unit_field(
            x=x,
            labels=labels,
            rival=rival_a,
            targets=targets
        )

        x1 = project_perturbation(
            x0=x0,
            x=x + h_t * _smooth_box_direction(u0),
            eps=cfg.eps,
            norm=cfg.norm,
            input_min=cfg.input_min,
            input_max=cfg.input_max
        )

        v1, margin_b = self._unit_field(
            x=x1,
            labels=labels,
            rival=rival_b,
            targets=targets
        )

        x2 = project_perturbation(
            x0=x0,
            x=x1 + h_t * _smooth_box_direction(v1),
            eps=cfg.eps,
            norm=cfg.norm,
            input_min=cfg.input_min,
            input_max=cfg.input_max
        )

        u2, _ = self._unit_field(
            x=x2,
            labels=labels,
            rival=rival_a,
            targets=targets
        )

        x3 = project_perturbation(
            x0=x0,
            x=x2 - h_t * _smooth_box_direction(u2),
            eps=cfg.eps,
            norm=cfg.norm,
            input_min=cfg.input_min,
            input_max=cfg.input_max
        )

        v3, _ = self._unit_field(
            x=x3,
            labels=labels,
            rival=rival_b,
            targets=targets
        )

        x4 = project_perturbation(
            x0=x0,
            x=x3 - h_t * _smooth_box_direction(v3),
            eps=cfg.eps,
            norm=cfg.norm,
            input_min=cfg.input_min,
            input_max=cfg.input_max
        )

        displacement = (x4 - x) / tf.maximum(
            h_t * h_t,
            tf.cast(1e-12, x.dtype)
        )

        if cfg.orthogonalize_bracket_against_first_order_fields:
            displacement = _remove_components(displacement, u0, v1)

        info = {
            "u0": u0,
            "v0": v1,
            "margin_a": margin_a,
            "margin_b": margin_b,
            "bracket_rms": tf.reshape(_rms_scale(displacement), [-1]),
            "rival_a": rival_a,
            "rival_b": rival_b,
        }

        return tf.stop_gradient(displacement), info

    def _step_size(self, step: int) -> float:
        cfg = self.cfg

        if cfg.steps <= 1:
            return float(cfg.alpha)

        if cfg.step_schedule == "constant":
            return float(cfg.alpha)

        if cfg.step_schedule == "sqrt":
            return float(cfg.alpha) / math.sqrt(1.0 + step)

        return float(cfg.alpha) * (
            0.10
            + 0.90
            * 0.5
            * (1.0 + math.cos(math.pi * step / max(cfg.steps - 1, 1)))
        )

    def _first_order_leak_weight(self, step: int) -> float:
        cfg = self.cfg

        if cfg.pure_bracket_mode:
            return 0.0

        if cfg.steps <= 1:
            return float(cfg.first_order_leak_end)

        p = float(step) / float(cfg.steps - 1)

        return (
            (1.0 - p) * float(cfg.first_order_leak_start)
            + p * float(cfg.first_order_leak_end)
        )

    def _direction_from_pair(
        self,
        x0: tf.Tensor,
        x: tf.Tensor,
        labels: tf.Tensor,
        targets: Optional[tf.Tensor],
        step: int,
        rival_a: tf.Tensor,
        rival_b: tf.Tensor,
    ) -> Tuple[tf.Tensor, Dict[str, tf.Tensor]]:
        """
        Compute CA-NCHA direction for one candidate class pair.
        """
        cfg = self.cfg

        if cfg.use_commutator_loop:
            bracket, info = self._commutator_loop_displacement(
                x0=x0,
                x=x,
                labels=labels,
                rival_a=rival_a,
                rival_b=rival_b,
                targets=targets
            )
        else:
            bracket, info = self._finite_difference_lie_bracket(
                x0=x0,
                x=x,
                labels=labels,
                rival_a=rival_a,
                rival_b=rival_b,
                targets=targets
            )

        leak = self._first_order_leak_weight(step)

        if leak > 0:
            active = 0.5 * (info["u0"] + info["v0"])
            direction = (
                (1.0 - leak) * _normalize_rms(bracket)
                + leak * _normalize_rms(active)
            )
        else:
            direction = bracket

        if cfg.fallback_to_active_field_if_bracket_degenerate:
            active = 0.5 * (info["u0"] + info["v0"])

            deg = tf.reshape(
                info["bracket_rms"] < tf.cast(cfg.degenerate_bracket_rms, x.dtype),
                [-1, 1, 1, 1]
            )

            direction = tf.where(deg, active, direction)
            info["fallback_used"] = tf.reshape(tf.cast(deg, tf.float32), [-1])

        else:
            info["fallback_used"] = tf.zeros(
                [tf.shape(x)[0]],
                dtype=tf.float32
            )

        info["direction_rms"] = tf.reshape(_rms_scale(direction), [-1])

        return tf.stop_gradient(direction), info

    def _pair_score(
        self,
        x0: tf.Tensor,
        x: tf.Tensor,
        direction: tf.Tensor,
        labels: tf.Tensor,
        targets: Optional[tf.Tensor],
        step: int,
        info: Dict[str, tf.Tensor]
    ) -> tf.Tensor:
        """
        Score a candidate class pair.

        The score is not used as a gradient.
        It only selects which pair's bracket direction to use.
        """
        mode = self.cfg.active_pair_score

        bracket_rms = info["bracket_rms"]
        margin_pressure = tf.maximum(info["margin_a"], info["margin_b"])
        fallback_used = info["fallback_used"]

        if mode == "bracket_rms":
            return bracket_rms

        if mode == "margin":
            return margin_pressure

        if mode == "lookahead_margin":
            alpha_t = self._step_size(step)

            _, lookahead_margin = self._line_search_update(
                x0=x0,
                x=x,
                direction=direction,
                labels=labels,
                targets=targets,
                alpha_t=alpha_t
            )

            return lookahead_margin

        # Balanced default:
        #   prefer non-degenerate brackets near active class boundaries.
        nonfallback_bonus = 1.0 - fallback_used

        return (
            tf.math.log1p(tf.maximum(bracket_rms, 0.0))
            + 0.15 * tf.nn.tanh(margin_pressure)
            + 0.05 * nonfallback_bonus
        )

    def _proposal_direction(
        self,
        x0: tf.Tensor,
        x: tf.Tensor,
        labels: tf.Tensor,
        targets: Optional[tf.Tensor],
        step: int,
    ) -> Tuple[tf.Tensor, Dict[str, tf.Tensor]]:
        """
        Top-k class-adaptive pair selection.

        For every sample:
            top-k active classes are selected.
            candidate class pairs are built.
            a bracket direction is computed for each pair.
            the best pair is selected adaptively.
        """
        cfg = self.cfg

        topk_classes = self._active_topk_classes(
            x=x,
            labels=labels,
            targets=targets
        )

        k_eff = int(topk_classes.shape[1])

        if cfg.active_pair_strategy in ("top2", "original"):
            pair_positions = [(0, 1)]
        else:
            pair_positions = _candidate_pair_positions(
                k_eff=k_eff,
                max_pairs=cfg.active_max_pairs
            )

        candidate_directions: List[tf.Tensor] = []
        candidate_scores: List[tf.Tensor] = []
        candidate_infos: List[Dict[str, tf.Tensor]] = []

        for pos_a, pos_b in pair_positions:
            rival_a = topk_classes[:, pos_a]
            rival_b = topk_classes[:, pos_b]

            direction, info = self._direction_from_pair(
                x0=x0,
                x=x,
                labels=labels,
                targets=targets,
                step=step,
                rival_a=rival_a,
                rival_b=rival_b
            )

            score = self._pair_score(
                x0=x0,
                x=x,
                direction=direction,
                labels=labels,
                targets=targets,
                step=step,
                info=info
            )

            candidate_directions.append(direction)
            candidate_scores.append(score)
            candidate_infos.append(info)

        direction_stack = tf.stack(candidate_directions, axis=1)
        score_stack = tf.stack(candidate_scores, axis=1)

        best_pair_index = tf.argmax(
            score_stack,
            axis=1,
            output_type=tf.int32
        )

        chosen_direction = tf.gather(
            direction_stack,
            best_pair_index,
            axis=1,
            batch_dims=1
        )

        def gather_vector_info(key: str) -> tf.Tensor:
            stacked = tf.stack([inf[key] for inf in candidate_infos], axis=1)

            return tf.gather(
                stacked,
                best_pair_index,
                axis=1,
                batch_dims=1
            )

        chosen_info: Dict[str, tf.Tensor] = {}

        for key in [
            "margin_a",
            "margin_b",
            "bracket_rms",
            "rival_a",
            "rival_b",
            "fallback_used",
            "direction_rms",
        ]:
            chosen_info[key] = gather_vector_info(key)

        chosen_info["candidate_pair_score"] = tf.gather(
            score_stack,
            best_pair_index,
            axis=1,
            batch_dims=1
        )

        chosen_info["candidate_pair_index"] = best_pair_index

        chosen_info["num_candidate_pairs"] = tf.fill(
            [tf.shape(x)[0]],
            tf.cast(len(pair_positions), tf.int32)
        )

        chosen_info["active_top_k"] = tf.fill(
            [tf.shape(x)[0]],
            tf.cast(k_eff, tf.int32)
        )

        return tf.stop_gradient(chosen_direction), chosen_info

    def _apply_direction(
        self,
        x0: tf.Tensor,
        x: tf.Tensor,
        direction: tf.Tensor,
        signed_alpha: float
    ) -> tf.Tensor:
        cfg = self.cfg

        a = tf.cast(signed_alpha, x.dtype)
        eps_t = tf.cast(cfg.eps, x.dtype)

        if cfg.norm.lower() in ("linf", "l_inf", "inf"):
            dx = eps_t * a * _smooth_box_direction(direction)

        elif cfg.norm.lower() in ("l2", "2"):
            dx = eps_t * a * _normalize_l2(direction)

        else:
            raise ValueError(f"Unsupported norm: {cfg.norm}")

        return project_perturbation(
            x0=x0,
            x=x + dx,
            eps=cfg.eps,
            norm=cfg.norm,
            input_min=cfg.input_min,
            input_max=cfg.input_max
        )

    def _line_search_update(
        self,
        x0: tf.Tensor,
        x: tf.Tensor,
        direction: tf.Tensor,
        labels: tf.Tensor,
        targets: Optional[tf.Tensor],
        alpha_t: float,
    ) -> Tuple[tf.Tensor, tf.Tensor]:
        """
        Try +/- bracket direction and several step factors.
        Keep the candidate with best attack margin.
        """
        scores, _ = self._scores_probs(x)

        best_margin = attack_margin_from_scores(
            scores=scores,
            labels=labels,
            targeted=self.cfg.targeted,
            targets=targets
        )

        best_x = x

        signed_steps: List[float] = []

        for f in list(self.cfg.line_search_factors):
            signed_steps.append(float(alpha_t) * float(f))

            if self.cfg.try_both_bracket_signs:
                signed_steps.append(-float(alpha_t) * float(f))

        for s in signed_steps:
            cand = self._apply_direction(
                x0=x0,
                x=x,
                direction=direction,
                signed_alpha=s
            )

            cand_scores, _ = self._scores_probs(cand)

            cand_margin = attack_margin_from_scores(
                scores=cand_scores,
                labels=labels,
                targeted=self.cfg.targeted,
                targets=targets
            )

            better = cand_margin > best_margin

            best_x = tf.where(
                tf.reshape(better, [-1, 1, 1, 1]),
                cand,
                best_x
            )

            best_margin = tf.where(
                better,
                cand_margin,
                best_margin
            )

        return tf.stop_gradient(best_x), tf.stop_gradient(best_margin)

    def _initial_adv(self, x0: tf.Tensor) -> tf.Tensor:
        cfg = self.cfg

        if not cfg.random_start:
            return tf.identity(x0)

        eps_t = tf.cast(cfg.eps, x0.dtype)

        if cfg.norm.lower() in ("linf", "l_inf", "inf"):
            delta = tf.random.uniform(
                tf.shape(x0),
                minval=-eps_t,
                maxval=eps_t,
                dtype=x0.dtype
            )

        elif cfg.norm.lower() in ("l2", "2"):
            delta = tf.random.normal(tf.shape(x0), dtype=x0.dtype)
            delta = _normalize_l2(delta)

            dim = tf.cast(
                tf.reduce_prod(tf.shape(x0)[1:]),
                x0.dtype
            )

            radius = eps_t * tf.pow(
                tf.random.uniform([tf.shape(x0)[0]], dtype=x0.dtype),
                1.0 / dim
            )

            delta = delta * tf.reshape(radius, [-1, 1, 1, 1])

        else:
            raise ValueError(f"Unsupported norm: {cfg.norm}")

        return project_perturbation(
            x0=x0,
            x=x0 + delta,
            eps=cfg.eps,
            norm=cfg.norm,
            input_min=cfg.input_min,
            input_max=cfg.input_max
        )

    def __call__(
        self,
        images: TensorLike,
        labels: TensorLike,
        targets: Optional[TensorLike] = None,
        return_info: bool = False,
    ) -> Union[tf.Tensor, Tuple[tf.Tensor, Dict[str, Any]]]:
        cfg = self.cfg

        x_base = tf.cast(tf.convert_to_tensor(images), tf.float32)
        y_base = labels_to_int(labels)
        t_base = None if targets is None else labels_to_int(targets)

        if cfg.targeted and t_base is None:
            raise ValueError("targets must be provided when targeted=True")

        b = tf.shape(x_base)[0]
        h = tf.shape(x_base)[1]
        w = tf.shape(x_base)[2]
        c = tf.shape(x_base)[3]

        r = max(int(cfg.restarts), 1)

        clean_scores, _ = self._scores_probs(x_base)
        clean_pred = tf.argmax(clean_scores, axis=1, output_type=tf.int32)
        clean_correct = tf.equal(clean_pred, y_base)

        x0 = tf.repeat(x_base, repeats=r, axis=0)
        labels_rep = tf.repeat(y_base, repeats=r, axis=0)
        targets_rep = None if t_base is None else tf.repeat(t_base, repeats=r, axis=0)

        adv = self._initial_adv(x0)
        best_adv = tf.identity(adv)
        best_margin = tf.fill([tf.shape(adv)[0]], tf.cast(-1e9, adv.dtype))

        margin_history: List[float] = []
        bracket_rms_history: List[float] = []
        direction_rms_history: List[float] = []
        fallback_rate_history: List[float] = []
        pair_score_history: List[float] = []
        active_top_k_history: List[float] = []
        num_candidate_pairs_history: List[float] = []

        rival_a_history: List[np.ndarray] = []
        rival_b_history: List[np.ndarray] = []
        pair_index_history: List[np.ndarray] = []

        for step in range(max(int(cfg.steps), 1)):
            direction, step_info = self._proposal_direction(
                x0=x0,
                x=adv,
                labels=labels_rep,
                targets=targets_rep,
                step=step
            )

            alpha_t = self._step_size(step)

            adv, margin_after = self._line_search_update(
                x0=x0,
                x=adv,
                direction=direction,
                labels=labels_rep,
                targets=targets_rep,
                alpha_t=alpha_t
            )

            better = margin_after > best_margin

            best_adv = tf.where(
                tf.reshape(better, [-1, 1, 1, 1]),
                adv,
                best_adv
            )

            best_margin = tf.where(
                better,
                margin_after,
                best_margin
            )

            if cfg.keep_history:
                margin_history.append(_safe_float(tf.reduce_mean(margin_after)))
                bracket_rms_history.append(_safe_float(tf.reduce_mean(step_info["bracket_rms"])))
                direction_rms_history.append(_safe_float(tf.reduce_mean(step_info["direction_rms"])))
                fallback_rate_history.append(_safe_float(tf.reduce_mean(step_info["fallback_used"])))
                pair_score_history.append(_safe_float(tf.reduce_mean(step_info["candidate_pair_score"])))
                active_top_k_history.append(_safe_float(tf.reduce_mean(tf.cast(step_info["active_top_k"], tf.float32))))
                num_candidate_pairs_history.append(_safe_float(tf.reduce_mean(tf.cast(step_info["num_candidate_pairs"], tf.float32))))

                try:
                    rival_a_history.append(step_info["rival_a"].numpy())
                    rival_b_history.append(step_info["rival_b"].numpy())
                    pair_index_history.append(step_info["candidate_pair_index"].numpy())
                except Exception:
                    pass

        best_adv_r = tf.reshape(best_adv, [b, r, h, w, c])
        best_margin_r = tf.reshape(best_margin, [b, r])

        best_idx = tf.argmax(
            best_margin_r,
            axis=1,
            output_type=tf.int32
        )

        chosen_adv = tf.gather(
            best_adv_r,
            best_idx,
            axis=1,
            batch_dims=1
        )

        chosen_adv = tf.stop_gradient(chosen_adv)

        final_scores, final_probs = self._scores_probs(chosen_adv)
        final_pred = tf.argmax(final_scores, axis=1, output_type=tf.int32)

        final_margin = attack_margin_from_scores(
            scores=final_scores,
            labels=y_base,
            targeted=cfg.targeted,
            targets=t_base
        )

        success = self._success(
            scores=final_scores,
            labels=y_base,
            targets=t_base
        )

        delta = chosen_adv - x_base

        if cfg.norm.lower() in ("linf", "l_inf", "inf"):
            norm_used = tf.reduce_max(tf.abs(delta), axis=[1, 2, 3])
        else:
            norm_used = _flat_l2_norm(delta)

        if not return_info:
            return chosen_adv

        clean_correct_f = tf.cast(clean_correct, tf.float32)

        denom = tf.maximum(
            tf.reduce_sum(clean_correct_f),
            1.0
        )

        success_clean_correct = (
            tf.reduce_sum(tf.cast(success, tf.float32) * clean_correct_f)
            / denom
        )

        info: Dict[str, Any] = {
            "attack_name": "CA-NCHA",
            "full_name": "Class-Adaptive Non-Commutative Holonomy Attack",
            "principle": "top-k adaptive Lie bracket / holonomy drift of active class-margin vector fields",
            "active_pair_strategy": cfg.active_pair_strategy,
            "active_top_k": cfg.active_top_k,
            "active_max_pairs": cfg.active_max_pairs,
            "active_pair_score": cfg.active_pair_score,

            "success": success.numpy(),
            "success_rate_all": _safe_float(tf.reduce_mean(tf.cast(success, tf.float32))),
            "success_rate_clean_correct": _safe_float(success_clean_correct),

            "clean_correct": clean_correct.numpy(),
            "clean_pred": clean_pred.numpy(),
            "adv_pred": final_pred.numpy(),

            "margin": final_margin.numpy(),
            "mean_margin": _safe_float(tf.reduce_mean(final_margin)),
            "norm_used": norm_used.numpy(),

            "best_restart_index": best_idx.numpy(),
            "restart_margins": best_margin_r.numpy(),
            "final_probabilities": final_probs.numpy(),

            "margin_history": margin_history,
            "bracket_rms_history": bracket_rms_history,
            "direction_rms_history": direction_rms_history,
            "fallback_rate_history": fallback_rate_history,
            "pair_score_history": pair_score_history,
            "active_top_k_history": active_top_k_history,
            "num_candidate_pairs_history": num_candidate_pairs_history,

            "rival_a_history": rival_a_history,
            "rival_b_history": rival_b_history,
            "pair_index_history": pair_index_history,

            "config": dict(cfg.__dict__),
        }

        return chosen_adv, info

    def attack_numpy(
        self,
        images: np.ndarray,
        labels: np.ndarray,
        targets: Optional[np.ndarray] = None,
        return_info: bool = True,
    ) -> Union[np.ndarray, Tuple[np.ndarray, Dict[str, Any]]]:
        result = self(
            images=images,
            labels=labels,
            targets=targets,
            return_info=return_info
        )

        if return_info:
            adv, info = result
            return adv.numpy(), info

        return result.numpy()

In [ ]:
# ============================================================
# Keras utilities
# ============================================================

def make_logits_model_from_softmax_model(model: tf.keras.Model) -> tf.keras.Model:
    """
    Convert a final Dense softmax model into an equivalent logits model.

    Works for models ending with:
        Dense(num_classes, activation='softmax')
    """
    last = model.layers[-1]

    if not isinstance(last, tf.keras.layers.Dense):
        raise ValueError("Expected final layer to be tf.keras.layers.Dense.")

    weights = last.get_weights()

    if not weights:
        raise ValueError("Final Dense layer has no weights. Build/load the model first.")

    logits_layer = tf.keras.layers.Dense(
        last.units,
        activation=None,
        use_bias=last.use_bias,
        name=f"{last.name}_linear_logits",
    )

    logits = logits_layer(last.input)

    logits_model = tf.keras.Model(
        model.inputs,
        logits,
        name=f"{model.name}_linear_logits"
    )

    logits_layer.set_weights(weights)
    logits_model.trainable = False

    return logits_model


def choose_input_domain_and_budget(
    x: np.ndarray,
    mode: str = "mobilenet_preprocessed",
    raw_eps: float = 8 / 255
) -> Tuple[float, float, float, str]:
    """
    Choose valid input range and eps.

    For MobileNet preprocess_input:
        raw [0,1] eps 8/255 becomes [-1,1] eps 16/255.
    """
    x_min = float(np.nanmin(x))
    x_max = float(np.nanmax(x))
    mode = mode.lower().strip()

    if mode == "zero_one":
        return 0.0, 1.0, raw_eps, "[0, 1]"

    if mode == "zero_255":
        return 0.0, 255.0, raw_eps * 255.0, "[0, 255]"

    if mode == "mobilenet_preprocessed":
        return -1.0, 1.0, 2.0 * raw_eps, "MobileNet [-1, 1]"

    if mode != "auto":
        raise ValueError(
            "mode must be 'auto', 'zero_one', 'zero_255', or 'mobilenet_preprocessed'."
        )

    if x_min >= -1.05 and x_max <= 1.05:
        if x_min < -0.05:
            return -1.0, 1.0, 2.0 * raw_eps, "auto: MobileNet [-1, 1]"

        return 0.0, 1.0, raw_eps, "auto: [0, 1]"

    if x_min >= -1.0 and x_max <= 255.5:
        return 0.0, 255.0, raw_eps * 255.0, "auto: [0, 255]"

    rng = max(x_max - x_min, 1.0)
    return x_min, x_max, raw_eps * rng, "auto: custom range"


def choose_least_likely_targets(
    model: tf.keras.Model,
    x: np.ndarray,
    labels: np.ndarray,
    batch_size: int = 64
) -> np.ndarray:
    """
    For targeted attack:
    choose least-likely class different from true label.
    """
    probs = model.predict(x, batch_size=batch_size, verbose=0)

    targets = np.argmin(probs, axis=1).astype("int32")
    labels = labels.astype("int32")

    pred_order = np.argsort(probs, axis=1)

    for i in range(len(targets)):
        if targets[i] == labels[i]:
            for cand in pred_order[i]:
                if cand != labels[i]:
                    targets[i] = int(cand)
                    break

    return targets


# ============================================================
# Kaggle runner for CA-NCHA
# ============================================================

def run_ca_ncha_on_kaggle_variables(
    model1: tf.keras.Model,
    X_test: np.ndarray,
    y_test_one_hot: np.ndarray,

    attack_n: int = 16,
    adv_batch_size: int = 1,

    targeted_attack: bool = False,

    input_domain_mode: str = "mobilenet_preprocessed",
    raw_pixel_eps: float = 8 / 255,

    use_logits_conversion: bool = True,
    pred_batch_size: int = 32,

    strong_mode: bool = True,
    pure_bracket_mode: bool = True,

    # Top-k adaptive controls
    active_top_k: int = 5,
    active_max_pairs: int = 6,
    active_pair_score: str = "bracket_margin",

    # For very strict bracket-only experiments, set this False.
    # For practical success, True is safer.
    fallback_to_active_field_if_bracket_degenerate: bool = True,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    Drop-in Kaggle runner.

    Expected:
        model1
        X_test
        y_test_one_hot

    For multi-class:
        model1.output_shape[-1] should be num_classes.
    """
    x_all = np.asarray(X_test).astype("float32")
    y_raw = np.asarray(y_test_one_hot)

    if y_raw.ndim >= 2:
        y_all = np.argmax(y_raw, axis=1).astype("int32")
    else:
        y_all = y_raw.reshape(-1).astype("int32")

    if x_all.shape[0] != y_all.shape[0]:
        raise ValueError(
            f"X/y size mismatch: {x_all.shape[0]} images vs {y_all.shape[0]} labels"
        )

    num_classes = int(model1.output_shape[-1])

    n_attack = min(int(attack_n), x_all.shape[0])

    x_attack = x_all[:n_attack]
    y_attack = y_all[:n_attack]

    input_min, input_max, eps, domain_name = choose_input_domain_and_budget(
        x=x_attack,
        mode=input_domain_mode,
        raw_eps=raw_pixel_eps
    )

    attack_model = model1
    outputs_are_probs = True
    logits_status = "using original softmax outputs as probabilities"

    if use_logits_conversion:
        try:
            attack_model = make_logits_model_from_softmax_model(model1)
            outputs_are_probs = False
            logits_status = "using logits-converted model"
        except Exception as exc:
            logits_status = f"logits conversion failed; using original softmax outputs: {repr(exc)}"

    valid_rivals = max(num_classes - 1, 1)

    if valid_rivals <= 1:
        effective_top_k = 2
    else:
        effective_top_k = min(max(2, int(active_top_k)), valid_rivals)

    if strong_mode:
        cfg = CANCHAConfig(
            eps=eps,
            norm="linf",
            input_min=input_min,
            input_max=input_max,
            model_outputs_are_probabilities=outputs_are_probs,
            targeted=targeted_attack,

            steps=80,
            restarts=3,
            alpha=0.72,
            random_start=True,

            bracket_probe_scale=0.35,
            use_commutator_loop=False,
            loop_probe_scale=0.25,
            orthogonalize_bracket_against_first_order_fields=True,

            pure_bracket_mode=pure_bracket_mode,
            first_order_leak_start=0.0,
            first_order_leak_end=0.0 if pure_bracket_mode else 0.12,

            try_both_bracket_signs=True,
            line_search_factors=(1.0, 0.5, 0.25),

            fallback_to_active_field_if_bracket_degenerate=fallback_to_active_field_if_bracket_degenerate,
            degenerate_bracket_rms=1e-8,

            active_pair_strategy="adaptive_topk",
            active_top_k=effective_top_k,
            active_max_pairs=int(active_max_pairs),
            active_pair_score=active_pair_score,

            step_schedule="cosine",
            keep_history=True,
        )

    else:
        cfg = CANCHAConfig(
            eps=eps,
            norm="linf",
            input_min=input_min,
            input_max=input_max,
            model_outputs_are_probabilities=outputs_are_probs,
            targeted=targeted_attack,

            steps=35,
            restarts=1,
            alpha=0.60,
            random_start=True,

            bracket_probe_scale=0.30,
            use_commutator_loop=False,
            orthogonalize_bracket_against_first_order_fields=True,

            pure_bracket_mode=pure_bracket_mode,

            try_both_bracket_signs=True,
            line_search_factors=(1.0, 0.5),

            fallback_to_active_field_if_bracket_degenerate=fallback_to_active_field_if_bracket_degenerate,
            degenerate_bracket_rms=1e-8,

            active_pair_strategy="adaptive_topk",
            active_top_k=effective_top_k,
            active_max_pairs=int(active_max_pairs),
            active_pair_score=active_pair_score,

            step_schedule="cosine",
            keep_history=True,
        )

    target_all = None

    if targeted_attack:
        target_all = choose_least_likely_targets(
            model=model1,
            x=x_attack,
            labels=y_attack,
            batch_size=pred_batch_size
        )

    attack = ClassAdaptiveNCHA(attack_model, cfg)

    adv_batches: List[np.ndarray] = []
    info_batches: List[Dict[str, Any]] = []

    batch_size = max(1, int(adv_batch_size))

    print("\nStarting CA-NCHA attack")
    print("num_classes:", num_classes)
    print("domain:", domain_name)
    print("input min/max detected:", float(np.min(x_attack)), float(np.max(x_attack)))
    print("eps:", eps)
    print("logits:", logits_status)
    print("steps:", cfg.steps)
    print("restarts:", cfg.restarts)
    print("pure_bracket_mode:", cfg.pure_bracket_mode)
    print("fallback_if_degenerate:", cfg.fallback_to_active_field_if_bracket_degenerate)
    print("active_top_k:", cfg.active_top_k)
    print("active_max_pairs:", cfg.active_max_pairs)
    print("active_pair_score:", cfg.active_pair_score)
    print("principle: top-k adaptive Lie bracket / holonomy drift of active class-margin fields")

    if num_classes <= 2:
        print("\nBinary-class note:")
        print("Only one rival class exists, so strict pairwise brackets can degenerate.")
        print("For true top-k adaptive behavior, use a model with 3 or more classes.\n")

    for start in range(0, n_attack, batch_size):
        end = min(start + batch_size, n_attack)

        xb = x_attack[start:end]
        yb = y_attack[start:end]
        tb = None if target_all is None else target_all[start:end]

        adv_b, info_b = attack.attack_numpy(
            images=xb,
            labels=yb,
            targets=tb,
            return_info=True
        )

        adv_batches.append(adv_b)
        info_batches.append(info_b)

        fallback_hist = info_b.get("fallback_rate_history", [])
        bracket_hist = info_b.get("bracket_rms_history", [])
        pair_score_hist = info_b.get("pair_score_history", [])

        fallback_mean = float(np.mean(fallback_hist)) if len(fallback_hist) else 0.0
        bracket_mean = float(np.mean(bracket_hist)) if len(bracket_hist) else 0.0
        pair_score_mean = float(np.mean(pair_score_hist)) if len(pair_score_hist) else 0.0

        print(
            f"batch {start:04d}:{end:04d} | "
            f"success_all={info_b['success_rate_all']:.4f} | "
            f"success_clean_correct={info_b['success_rate_clean_correct']:.4f} | "
            f"mean_margin={info_b['mean_margin']:.4f} | "
            f"mean_norm={float(np.mean(info_b['norm_used'])):.6f} | "
            f"bracket_rms={bracket_mean:.4e} | "
            f"fallback={fallback_mean:.3f} | "
            f"pair_score={pair_score_mean:.4f}"
        )

    adv_images = np.concatenate(adv_batches, axis=0)

    clean_probs = model1.predict(
        x_attack,
        batch_size=pred_batch_size,
        verbose=0
    )

    adv_probs = model1.predict(
        adv_images,
        batch_size=pred_batch_size,
        verbose=0
    )

    clean_pred = np.argmax(clean_probs, axis=1)
    adv_pred = np.argmax(adv_probs, axis=1)

    clean_correct = clean_pred == y_attack

    if targeted_attack:
        success_all = adv_pred == target_all
    else:
        success_all = adv_pred != y_attack

    delta = adv_images - x_attack
    linf = np.max(np.abs(delta), axis=(1, 2, 3))

    all_fallback_values: List[float] = []
    all_bracket_values: List[float] = []
    all_pair_score_values: List[float] = []
    all_num_pairs_values: List[float] = []

    for info_b in info_batches:
        all_fallback_values.extend(info_b.get("fallback_rate_history", []))
        all_bracket_values.extend(info_b.get("bracket_rms_history", []))
        all_pair_score_values.extend(info_b.get("pair_score_history", []))
        all_num_pairs_values.extend(info_b.get("num_candidate_pairs_history", []))

    mean_fallback_rate = float(np.mean(all_fallback_values)) if len(all_fallback_values) else 0.0
    mean_bracket_rms = float(np.mean(all_bracket_values)) if len(all_bracket_values) else 0.0
    mean_pair_score = float(np.mean(all_pair_score_values)) if len(all_pair_score_values) else 0.0
    mean_num_candidate_pairs = float(np.mean(all_num_pairs_values)) if len(all_num_pairs_values) else 0.0

    final_info: Dict[str, Any] = {
        "attack_name": "CA-NCHA",
        "full_name": "Class-Adaptive Non-Commutative Holonomy Attack",
        "principle": "top-k adaptive Lie bracket / holonomy drift of active class-margin vector fields",

        "n_attack": int(n_attack),
        "num_classes": int(num_classes),
        "targeted": bool(targeted_attack),
        "target_labels": target_all,

        "input_domain": domain_name,
        "eps": float(eps),

        "steps": int(cfg.steps),
        "restarts": int(cfg.restarts),
        "pure_bracket_mode": bool(cfg.pure_bracket_mode),

        "active_pair_strategy": cfg.active_pair_strategy,
        "active_top_k": int(cfg.active_top_k),
        "active_max_pairs": int(cfg.active_max_pairs),
        "active_pair_score": cfg.active_pair_score,
        "mean_num_candidate_pairs": mean_num_candidate_pairs,

        "fallback_to_active_field_if_bracket_degenerate": bool(cfg.fallback_to_active_field_if_bracket_degenerate),
        "mean_fallback_rate": mean_fallback_rate,
        "mean_bracket_rms": mean_bracket_rms,
        "mean_pair_score": mean_pair_score,

        "clean_accuracy_on_subset": float(np.mean(clean_correct)),
        "adversarial_accuracy_on_subset": float(np.mean(adv_pred == y_attack)),
        "attack_success_all_samples": float(np.mean(success_all)),
        "attack_success_clean_correct_only": (
            float(np.mean(success_all[clean_correct]))
            if np.any(clean_correct)
            else float("nan")
        ),

        "max_linf_perturbation": float(np.max(linf)),
        "mean_linf_perturbation": float(np.mean(linf)),

        "clean_pred": clean_pred,
        "adv_pred": adv_pred,
        "true_labels": y_attack,
        "success_per_sample": success_all,
        "linf_per_sample": linf,
        "batch_infos": info_batches,
        "config": dict(cfg.__dict__),
    }

    np.save("/kaggle/working/ca_ncha_adv_X_test.npy", adv_images)
    np.save("/kaggle/working/ca_ncha_clean_X_test_subset.npy", x_attack)
    np.save("/kaggle/working/ca_ncha_true_labels_subset.npy", y_attack)
    np.save("/kaggle/working/ca_ncha_clean_pred_subset.npy", clean_pred)
    np.save("/kaggle/working/ca_ncha_adv_pred_subset.npy", adv_pred)

    print("\n================ CA-NCHA RESULTS ================")
    print("Clean accuracy on attacked subset:", final_info["clean_accuracy_on_subset"])
    print("Adversarial accuracy on attacked subset:", final_info["adversarial_accuracy_on_subset"])
    print("Attack success, all samples:", final_info["attack_success_all_samples"])
    print("Attack success, clean-correct only:", final_info["attack_success_clean_correct_only"])
    print("Max Linf perturbation:", final_info["max_linf_perturbation"])
    print("Mean Linf perturbation:", final_info["mean_linf_perturbation"])
    print("Mean bracket RMS:", final_info["mean_bracket_rms"])
    print("Mean fallback rate:", final_info["mean_fallback_rate"])
    print("Mean candidate pair score:", final_info["mean_pair_score"])
    print("Mean number of candidate pairs:", final_info["mean_num_candidate_pairs"])
    print("Saved: /kaggle/working/ca_ncha_adv_X_test.npy")
    print("=================================================")

    return adv_images, final_info

In [ ]:
adv_images, ca_ncha_info = run_ca_ncha_on_kaggle_variables(
    model1=model1,
    X_test=X_test,
    y_test_one_hot=y_test_one_hot,

    attack_n=16,
    adv_batch_size=1,

    targeted_attack=False,

    input_domain_mode="mobilenet_preprocessed",
    raw_pixel_eps=8 / 255,

    use_logits_conversion=True,
    pred_batch_size=32,

    strong_mode=True,
    pure_bracket_mode=True,

    # Top-k class-adaptive pair selection
    active_top_k=5,

    # 0 means all top-k pairs.
    # For top_k=5, all pairs = 10.
    # Start with 6 for speed, then use 0 or 10 for stronger experiments.
    active_max_pairs=6,

    # Good default:
    active_pair_score="bracket_margin",

    fallback_to_active_field_if_bracket_degenerate=True,
)

In [ ]:
adv_images, ca_ncha_info = run_ca_ncha_on_kaggle_variables(
    model1=model1,
    X_test=X_test,
    y_test_one_hot=y_test_one_hot,

    attack_n=16,
    adv_batch_size=1,

    targeted_attack=False,
    input_domain_mode="mobilenet_preprocessed",
    raw_pixel_eps=8 / 255,

    use_logits_conversion=True,
    strong_mode=True,
    pure_bracket_mode=True,

    active_top_k=5,
    active_max_pairs=0,
    active_pair_score="lookahead_margin",

    fallback_to_active_field_if_bracket_degenerate=True,
)